# MPC → PPO: xác nhận công thức DAgger trên GPU (Colab)

Chạy lại **công thức DAgger** đã chốt trên CPU (`docs/mpc_ppo_swingup_recipe.md`) so
với **PPO thường**, 10 seed mới, trên GPU.

- Task: `Mjlab-Cartpole-Swingup`, 64 env PPO, 200 vòng, đánh giá mỗi 10 vòng (400 bước).
- **DAgger:** thầy MPOPI (K = 8 × 4 vòng, H = 20), không nhiễu; policy tự chạy trên 8 env
  phụ, MPC gắn nhãn; chỉ BC, trọng số 1,0 giảm về 0 sau 30 vòng; thu dữ liệu 20 vòng đầu.
- Seed **500–509** (chưa dùng trong nghiên cứu CPU).

**Giả thuyết đặt trước:** AUC của DAgger > AUC của PPO, kiểm định Wilcoxon cặp hai phía,
α = 0,05. Ngoài ra đo thời gian thực trên GPU để đạt 0,040.

**Trước khi chạy:** *Runtime → Change runtime type → GPU* (T4 là đủ). Nên bật
`SAVE_TO_DRIVE`: mỗi lần chạy lưu kết quả riêng, nếu Colab ngắt thì chạy lại mục 6, các
lần chạy đã xong sẽ được bỏ qua.

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpc-stage1"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown Seed và số vòng lặp (giữ nguyên để đúng giả thuyết đã đặt trước):
FIRST_SEED = 500  #@param {type:"integer"}
NUM_SEEDS = 10  #@param {type:"integer"}
ITERATIONS = 200  #@param {type:"integer"}
#@markdown Số lần chạy song song trên GPU (2 thường ổn với T4):
PARALLEL = 2  #@param {type:"integer"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpc_dagger_gpu"
else:
  OUT_ROOT = "/content/mpc_dagger_gpu"
os.makedirs(OUT_ROOT, exist_ok=True)
SEEDS = list(range(FIRST_SEED, FIRST_SEED + NUM_SEEDS))
print("Kết quả lưu tại:", OUT_ROOT, "| seed:", SEEDS)

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import subprocess


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

Nhánh `mpc-stage1` phải được push bản mới nhất lên GitHub trước (có commit
`Add noise-free and DAgger behavior cloning ...`).

In [ ]:
import shutil

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)

REQUIRED = [
  "src/mjlab/mpc/collector.py",
  "tests/test_mpc_ppo.py",
  "docs/mpc_ppo_swingup_recipe.md",
]
missing = [f for f in REQUIRED if not os.path.exists(f"{REPO_DIR}/{f}")]
config_src = open(f"{REPO_DIR}/src/mjlab/rl/mpopi/config.py").read()
if missing or "bc_floor" not in config_src:
  raise RuntimeError(
    "Code trên GitHub đã cũ (thiếu DAgger / bc_floor). "
    "Hãy push bản mới nhất của nhánh mpc-stage1 rồi chạy lại ô này."
  )
print("OK: code có công thức DAgger.")

## 3. Cài đặt môi trường

`uv sync` **không** kèm `--extra cpu` để có PyTorch bản CUDA. Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch; print('torch', torch.__version__, 'cuda', "
  "torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)

## 4. Test trên GPU

Các test của MPC và `mpc_ppo` tự chạy trên GPU khi có. Đây là lần đầu planner, bộ thu dữ
liệu và DAgger chạy trên GPU; nếu ô này lỗi thì **dừng lại**, đừng chạy các mục sau.

In [ ]:
run(
  "uv run pytest tests/test_mpc_sampling.py tests/test_mpc_ppo.py "
  "tests/test_mpopi_algorithm.py -q -p no:warnings",
  cwd=REPO_DIR,
)

## 5. Chạy thử nhanh (smoke test) và đo tốc độ

Một lần chạy DAgger ngắn (5 vòng, thu MPC 3 vòng) để kiểm tra đường chạy đầy đủ và ước
lượng thời gian.

In [ ]:
COMMON = (
  "uv run python scripts/benchmarks/mpopi_benchmark.py --task Mjlab-Cartpole-Swingup "
  "--device cuda:0 --num-envs 64 --eval-every 10 --eval-steps 400 --seeds 1 "
  "--progress-every 20"
)
DAGGER = (
  "--arms D_mpc_bc_only --mpc.execution-std 0 --mpc.driver policy "
  "--mpc.planner.num-samples 8 --mpc.planner.iterations 4"
)
ARMS = {"A_ppo": "--arms A_ppo", "dagger_mpopi": DAGGER}

import time

start = time.time()
run(
  f"{COMMON} --iterations 5 --seed-offset 0 {DAGGER} --mpc.collect-iterations 3 "
  f"--out-dir {OUT_ROOT}/smoke",
  cwd=REPO_DIR,
)
print(f"Smoke test xong sau {time.time() - start:.0f} s")

## 6. Chạy thí nghiệm chính

2 nhánh × 10 seed = 20 lần chạy, mỗi lần một tiến trình riêng, `PARALLEL` lần cùng lúc.
Lần chạy nào đã có `curves.csv` thì bỏ qua, nên có thể chạy lại ô này sau khi Colab ngắt.
Log từng lần chạy: `OUT_ROOT/logs/<nhánh>_s<seed>.log`.

In [ ]:
import concurrent.futures as cf

LOG_DIR = f"{OUT_ROOT}/logs"
os.makedirs(LOG_DIR, exist_ok=True)


def job(arm: str, seed: int) -> str:
  out = f"{OUT_ROOT}/{arm}/s{seed}"
  if os.path.exists(f"{out}/curves.csv"):
    return f"{arm} seed {seed}: đã có, bỏ qua"
  cmd = f"{COMMON} --iterations {ITERATIONS} --seed-offset {seed} {ARMS[arm]} --out-dir {out}"
  env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
  start = time.time()
  with open(f"{LOG_DIR}/{arm}_s{seed}.log", "w") as log:
    code_ = subprocess.call(cmd, shell=True, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, env=env)
  status = "xong" if code_ == 0 else f"LỖI (exit {code_}), xem log"
  return f"{arm} seed {seed}: {status} sau {time.time() - start:.0f} s"


jobs = [(arm, seed) for seed in SEEDS for arm in ARMS]
with cf.ThreadPoolExecutor(max_workers=PARALLEL) as pool:
  for msg in pool.map(lambda a: job(*a), jobs):
    print(msg, flush=True)

In [ ]:
#@title Xem tiến độ (chạy ô này trong lúc mục 6 đang chạy, từ một tab khác nếu cần)
import glob

for path in sorted(glob.glob(f"{OUT_ROOT}/logs/*.log")):
  lines = [l for l in open(path) if l.startswith("[")]
  print(os.path.basename(path), "|", lines[-1].strip()[:110] if lines else "đang khởi động / thu MPC")

## 7. Phân tích

AUC = điểm trung bình suốt quá trình train; so sánh cặp theo seed. Thời gian đạt 0,040
được ước lượng từ thời gian thu MPC thực đo cộng thời gian PPO trung bình mỗi vòng.

In [ ]:
import csv
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

THRESH, MAX = 0.040, 0.05


def load(arm):
  runs = {}
  for p in glob.glob(f"{OUT_ROOT}/{arm}/s*/curves.csv"):
    for r in csv.DictReader(open(p)):
      runs.setdefault(int(r["seed"]), []).append(r)
  return runs


def fl(x):
  try:
    return float(x)
  except ValueError:
    return math.nan


def run_stats(rows):
  ev = [(int(r["iteration"]), fl(r["eval_return"])) for r in rows]
  ev = [(i, s) for i, s in ev if not math.isnan(s)]
  scores = [s for _, s in ev]
  tail = max(1, len(scores) // 10)
  collect = [fl(r.get("mpc/collect_seconds", "nan")) for r in rows]
  collect = [0.0 if math.isnan(c) else c for c in collect]
  total = fl(rows[0]["run_seconds"])
  ppo_per_it = (total - sum(collect)) / len(rows)
  hit = next((i for i, s in ev if s >= THRESH), None)
  hit_s = None if hit is None else sum(collect[: hit + 1]) + ppo_per_it * (hit + 1)
  return dict(auc=sum(scores) / len(scores), final=sum(scores[-tail:]) / tail,
              hit=hit, hit_s=hit_s, seconds=total, mpc_s=sum(collect), curve=dict(ev))


table = {arm: {s: run_stats(r) for s, r in load(arm).items()} for arm in ARMS}
for arm, t in table.items():
  st = [t[s] for s in sorted(t)]
  if not st:
    print(arm, ": chưa có kết quả")
    continue
  print(f"{arm:>13}: n={len(st)}  AUC {np.mean([x['auc'] for x in st]):.4f}"
        f"  final {np.mean([x['final'] for x in st]):.4f}"
        f"  đạt 0.040: {sum(x['hit'] is not None for x in st)}/{len(st)} seed"
        f"  thời gian chạy {np.mean([x['seconds'] for x in st]):.0f} s (MPC {np.mean([x['mpc_s'] for x in st]):.0f} s)")
  print(f"{'':>15}vòng đạt 0.040: {[x['hit'] for x in st]}")
  print(f"{'':>15}giây đạt 0.040: {[None if x['hit_s'] is None else round(x['hit_s']) for x in st]}")

a, b = table["dagger_mpopi"], table["A_ppo"]
seeds = sorted(set(a) & set(b))
if len(seeds) >= 2:
  for m in ("auc", "final"):
    d = [a[s][m] - b[s][m] for s in seeds]
    p = stats.wilcoxon(d).pvalue if any(d) else math.nan
    print(f"\nDAgger − PPO [{m}]: {np.mean(d):+.4f}, {sum(x > 0 for x in d)}/{len(d)} seed dương, Wilcoxon p = {p:.3g}")
  verdict = "XÁC NHẬN" if np.mean([a[s]['auc'] - b[s]['auc'] for s in seeds]) > 0 and stats.wilcoxon([a[s]['auc'] - b[s]['auc'] for s in seeds]).pvalue < 0.05 else "CHƯA XÁC NHẬN"
  print(f"Giả thuyết (AUC DAgger > PPO, p < 0.05, {len(seeds)} seed): {verdict}")

fig, ax = plt.subplots(figsize=(7, 3.4))
for arm, color, label in (("A_ppo", "#555555", "PPO thường"), ("dagger_mpopi", "#d62728", "DAgger, thầy MPOPI")):
  t = table[arm]
  if not t:
    continue
  its = sorted(set.intersection(*(set(x["curve"]) for x in t.values())))
  y = np.array([[t[s]["curve"][i] / MAX for i in its] for s in t])
  ax.plot(its, y.mean(0), color=color, lw=2, label=f"{label} (TB {len(t)} seed)")
  ax.fill_between(its, y.min(0), y.max(0), color=color, alpha=0.15, lw=0)
ax.set_xlabel("Vòng lặp PPO")
ax.set_ylabel("Điểm chuẩn hóa")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.savefig(f"{OUT_ROOT}/dagger_vs_ppo_gpu.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Tải kết quả về

In [ ]:
shutil.make_archive("/content/mpc_dagger_gpu_results", "zip", OUT_ROOT)
from google.colab import files

files.download("/content/mpc_dagger_gpu_results.zip")